### Ссылки: туториалы и гайды
**Pytorch**
* [PyTorch Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html) - базовый туториал по PyTorch
* [PyTorch Recipes](https://docs.pytorch.org/tutorials/recipes_index.html) - примеры PyTorch

**Tranformers**
* [Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) - объяснение архитектуры трансформера с кодом
* [Transformers](https://huggingface.co/docs/transformers/index) - подробная документация по библиотеке Transformers
* [Transformers GitHub](https://github.com/huggingface/transformers) - исходный код библиотеки Transformers

**Advanced topics**
* [С++ API](https://docs.pytorch.org/tutorials/advanced/cpp_frontend.html)
* [Custom Kernels](https://mljourney.com/how-to-write-a-custom-cuda-kernel-for-pytorch/)
* [Torch compile](https://docs.pytorch.org/tutorials/intermediate/torch_compile_tutorial.html)

**Intro**
* [huggingface - intro](https://huggingface.co/learn/llm-course/chapter1/1)
* [Deep Learning - intro](https://colah.github.io/posts/2014-07-NLP-RNNs-Representations/)
* [neuralnetworksanddeeplearning.com](http://neuralnetworksanddeeplearning.com/chap1.html)
* [handbook-yandex](https://education.yandex.ru/handbook/ml/article/transformery)

**Visualizations**
* [A Neural Network Playground](https://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.28994&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false)
* [The Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/)
* [3Blue1Brown](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi)
  
**Notes**
* [course22p2](https://github.com/fastai/course22p2/tree/master/nbs)
* [Practical Deep Learning for Coders](https://course.fast.ai/)

### Recap
**Квантизация**
* Что такое квантизация?
* Для чего она нужна?
* Какие виды квантизации вы знаете?
* Какие сложности при работает с квантизацией вы знаете?

**Трансформер**
* Что такое трансформер? На каких данных его применяют?
* Как он устроен?

In [ ]:
import torch
import numpy as np
from matplotlib import pyplot as plt

import os
os.environ["http_proxy"] = "http://127.0.0.1:3128"
os.environ["https_proxy"] = "http://127.0.0.1:3128"

In [ ]:
# --- Создание тензоров ---
# Аналогия с numpy — почти 1:1
x_np = np.array([1, 2, 3])
x_torch = torch.tensor([1, 2, 3])

In [ ]:
# Из numpy и обратно (zero-copy!)
x = torch.from_numpy(x_np)
x.numpy()

In [ ]:
# dtype и device — два ключевых параметра
x = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32, device='cpu')
x_gpu = x.to('cuda')  # .to() — универсальный метод для перемещения между device/dtype

device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
# Частые паттерны
torch.zeros(3, 4)
torch.ones(2, 3)
torch.randn(2, 3)        # нормальное распределение
torch.randint(0, 10, (3,)) # случайные целые

In [ ]:
# --- Операции ---
# Всё как в numpy: +, *, matmul, reshape, permute, ...
a = torch.randn(2, 3)
b = torch.randn(3, 4)
c = a @ b              # матричное умножение
c = a.reshape(3, 2)    # reshape (может быть view — zero-copy)
c = a.view(3, 2)       # view — строго zero-copy (когда возможно)

In [ ]:
# Практический совет: всегда проверяй shape
print(a.shape, b.shape, c.shape)

# view vs reshape — view быстрее, но может упасть на не-contiguous тензорах
# - from_numpy / .numpy() — zero-copy, данные не копируются

### Задачки

**Реализовать self-attention**

Даны тензоры $Q$,$K$,$V$. (Не multi-head)

$\text{Attention}(Q,K,V) = \text{Softmax}(\frac{Q \cdot K^T}{\sqrt{d}})V$



In [ ]:
d = 2
Q = torch.randn(2, 2)
K = torch.randn(2, 2)
V = torch.randn(2, 2)
torch.nn.functional.softmax((Q @ K.T) / np.sqrt(d), dim=-1) @ V

In [ ]:
a = torch.randn(20000,20000).to("cuda:0")

In [ ]:
b = a@a

In [ ]:
a = a.detach().cpu().numpy()

In [ ]:
b = a@a

# Магия градиентов

**Проверка** 
* Как работает back propagation?
* Какие затраты по памяти (что мы храним и обязательно ли это)?

In [ ]:
x = torch.tensor([2.0], requires_grad=True) # — PyTorch запоминает все операции для backward pass
y = x ** 2 + 3 * x + 1  # y = x² + 3x + 1
y.backward() # — вычисляет градиенты по всему графу
print(x.grad)  # tensor([7.]) → dy/dx = 2x + 3 = 2*2 + 3 = 7

In [ ]:
# линейная регрессия "ручками" ---

hidden_dim = 100
w = torch.randn(hidden_dim, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
data_size = 1000
X = torch.randn(data_size, hidden_dim)

w_true = torch.randn(hidden_dim, 1, requires_grad=False)
b_true = torch.randn(1, requires_grad=False)
y_true = X @ w_true + b_true
loss_history = []
for step in range(350):
    y_pred = X @ w + b
    loss = ((y_pred - y_true) ** 2).mean()
    loss_history.append(loss.item())
     
    loss.backward() 
    with torch.no_grad(): # — контекстный менеджер для inference (не считает граф)
        w -= 0.01 * w.grad
        b -= 0.01 * b.grad
    w.grad.zero_() # — обязательно между шагами, иначе градиенты накапливаются
    b.grad.zero_()

print(f"w={(w-w_true).abs().mean().item():.3f}, b={(b-b_true).abs().mean().item():.3f}")
print(f"loss={loss_history[-1]}")
plt.plot(loss_history)
plt.grid()
plt.xlabel("Epochs")
plt.ylabel("Loss")

**Вопросы**
* Можно ли быть уверенным что наша модель работает? Как это исправавить?
* Что делать если data size очень большой?

# Построение модели

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

* Какие архитектуры нейросетей вы знаете?
* Какие виды задач вы знаете?
* Какие виды обучения вы значете?

In [ ]:
# модель для классификации текста 
class SentimentModel(nn.Module): # — контейнер: автоматически собирает параметры, обрабатывает device, поддерживает hooks
  def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2):
      super().__init__()
      self.embedding = nn.Embedding(vocab_size, embed_dim)
      self.fc1 = nn.Linear(embed_dim, hidden_dim)
      self.fc2 = nn.Linear(hidden_dim, num_classes)

  def forward(self, x): # Только forward() — backward PyTorch пишет сам
      # x: [batch, seq_len] → целые числа (индексы токенов)
      x = self.embedding(x)     # [batch, seq_len, embed_dim]
      x = x.mean(dim=1)         # [batch, embed_dim] — усреднили по seq
      x = F.relu(self.fc1(x))   # [batch, hidden_dim]
      x = self.fc2(x)           # [batch, num_classes]
      return x

model = SentimentModel(vocab_size=10000, embed_dim=64, hidden_dim=32)
print(model)                       # красивый дамп структуры

In [ ]:
print(sum(p.numel() for p in model.parameters()))  # число параметров. много?

* Сколько памяти нужно для обучения 1B модели?

In [ ]:
# --- state_dict: save / load ---
torch.save(model.state_dict(), 'model.pt') #  — словарь {имя: тензор}, универсальный формат для сохранения
model.load_state_dict(torch.load('model.pt', weights_only=True)) # — безопасность (предотвращает pickle-эксплойты)

#  Train loop + DataLoader 

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
# --- Dataset ---
class ReviewDataset(Dataset):
  def __init__(self, texts, labels, vocab, max_len=128):
      self.texts = texts
      self.labels = labels
      self.vocab = vocab
      self.max_len = max_len

  def __len__(self):
      return len(self.texts)

  def __getitem__(self, idx):
      tokens = [self.vocab.get(w, 1) for w in self.texts[idx].split()]
      tokens = tokens[:self.max_len]
      tokens = tokens + [0] * (self.max_len - len(tokens))  # padding
      return torch.tensor(tokens), torch.tensor(self.labels[idx])

In [ ]:
from datasets import load_dataset
imdb = load_dataset("stanfordnlp/imdb")
train_texts = imdb["train"]["text"]
train_labels = imdb["train"]["label"]
test_labels = imdb["test"]["label"]
test_texts = imdb["test"]["text"]

In [ ]:
print(f"Пример: {train_texts[0][:200]}... → label={train_labels[0]}")

In [ ]:
print(set(train_labels))

In [ ]:
from collections import Counter
# Строим словарь по обучающей выборке
''' Проходим по всем текстам, приводим их к нижнему регистру и разбиваем на отдельные слова. Counter считает, сколько раз каждое слово встретилось во всем наборе данных.
В реальных данных могут быть миллионы уникальных слов (опечатки, редкие имена). Чтобы модель не "раздувалась",оставляем только 9998 самых частых слов. Мы вычитаем 2, так как резервируем место под специальные токены.

Каждому частому слову присваивается уникальный номер (ID). Мы начинаем индексацию с 2, потому что индексы 0 и 1 зарезервированы под служебные цели:0 обычно используется для <pad> (заполнение коротких предложений до нужной длины).
1 обычно используется для <unk> (unknown — для слов, которые не попали в топ-10000). Получили "карту", по которой слово "the" превратится, например, в число 2, 
а любое редкое слово (которого нет в словаре) при обработке датасетом станет единицей.
'''
VOCAB_SIZE = 10000
counter = Counter()
for text in train_texts:
    counter.update(text.lower().split())

# 0 = <pad>, 1 = <unk> — поэтому берём VOCAB_SIZE-2 самых частых
most_common = counter.most_common(VOCAB_SIZE - 2)
vocab = {word: idx + 2 for idx, (word, _) in enumerate(most_common)}

print(f"Словарь: {len(vocab)} слов (первые 5: {list(vocab.items())[:5]})")

In [ ]:
# Создание датасета
dataset = ReviewDataset(train_texts, train_labels, vocab, max_len=128)
test_dataset = ReviewDataset(test_texts, train_labels, vocab, max_len=128)

In [ ]:
# --- DataLoader ---
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
# Проверка: достаем один батч
batch_x, batch_y = next(iter(train_loader))
print(f"Формат батча X: {batch_x.shape}") # Должно быть [32, 128]
print(f"Формат батча Y: {batch_y.shape}") # Должно быть [32]

# train loop 

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentimentModel(10000, 64, 32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
len(test_dataset)

In [ ]:
def evaluation(model):
    model.eval()
    score = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x.to(device))
            preds = logits.argmax(dim=1)
            score += (batch_y == preds).sum().item()
            
        accuracy = score / len(test_dataset)
        return accuracy

In [ ]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
    
        optimizer.zero_grad()          # 1. Обнуляем градиенты
        logits = model(batch_x)        # 2. Forward pass
        loss = criterion(logits, batch_y)  # 3. Считаем loss
        loss.backward()                # 4. Backward pass
        optimizer.step()               # 5. Обновляем веса
        
        total_loss += loss.item()
    accuracy = evaluation(model)
    print(f"Epoch {epoch}: loss={total_loss/len(train_loader):.4f}, {accuracy=:.4f}")

# Hugging Face

In [ ]:
# Hugging Face даёт вам готовую модель в 3 строчки кода, но важно понимать что под капотом — тот же PyTorch

In [ ]:
#  модель за 3 строчки
from transformers import pipeline

In [ ]:
# --- Sentiment pipeline — из коробки ---
classifier = pipeline("sentiment-analysis")
classifier("This movie was absolutely fantastic!")  # [{'label': 'POSITIVE', 'score': 0.999}]

In [ ]:
# --- Разные задачи — один API ---
generator = pipeline("text-generation", model="gpt2")
generator("Once upon a time", max_new_tokens=50)

In [ ]:
print(generator.model)

In [ ]:
print(sum(p.numel() for p in generator.model.parameters()))  # число параметров

In [ ]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
summarizer("Long article text here...")

In [ ]:
# pipeline = tokenizer + model + postprocessing 

# заглядываем под капот 

[GPT2](https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py#L413)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [ ]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

In [ ]:
# --- Config — архитектура модели ---
from transformers import AutoConfig
config = AutoConfig.from_pretrained(model_name)
print(config)  # hidden_size=768, num_attention_heads=12, num_hidden_layers=6, ...

In [ ]:
# --- Tokenizer — текст → числа ---
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokens = tokenizer("Hello world!", return_tensors="pt")
print(tokens)
# {'input_ids': tensor([[101, 7592, 2088, 999, 102]]),
#  'attention_mask': tensor([[1, 1, 1, 1, 1]])}

In [ ]:
torch.set_grad_enabled(False)

In [ ]:
# --- Model — числа → предсказания ---
model = AutoModelForSequenceClassification.from_pretrained(model_name).to("cuda:0").eval()
with torch.no_grad():
    tokens = {k: v.to("cuda:0") for k,v in tokens.items()}
    outputs = model(**tokens)
    # outputs.logits, outputs.loss (если есть labels)

In [ ]:
print(outputs.logits)
probs = torch.softmax(outputs.logits, dim=-1)
print(probs)  # tensor([[0.004, 0.996]]) → POSITIVE

In [ ]:
from torch.profiler import profile, ProfilerActivity, record_function
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], profile_memory=True, record_shapes=True) as prof:
    with record_function("model_inference"):
        outputs = model(**tokens)

In [ ]:
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))

In [ ]:
prof.export_chrome_trace("trace.json") # chrome://tracing

In [ ]:
import torch.quantization
quantized_model = torch.quantization.quantize_dynamic(model.to("cpu"), {torch.nn.Linear}, dtype=torch.qint8)
print(quantized_model)

In [ ]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    print('Size (MB):', os.path.getsize("temp.p")/1e6)
    os.remove('temp.p')

print_size_of_model(model)
print_size_of_model(quantized_model)

In [ ]:
with torch.no_grad():
    tokens = {k: v.to(model.device) for k,v in tokens.items()}
    outputs = quantized_model(**tokens)
print(outputs.logits)
probs = torch.softmax(outputs.logits, dim=-1)
print(probs)  # tensor([[0.004, 0.996]]) → POSITIVE

In [ ]:
# --- Смотрим на state_dict — знакомо? ---
print(model.state_dict().keys())
# Те же самые weight/bias тензоры, что и в PyTorch!

# Дообучение: PEFT / LoRA 

* Что такое transfer learning?
* Как работет Low-Rank adaptation? В чем ее преимущества

In [ ]:
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
# --- Обычное дообучение — дорого (все параметры) ---
# model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
# --- PEFT / LoRA — дёшево (0.1-1% параметров) ---
model = AutoModelForSequenceClassification.from_pretrained(model_name)

lora_config = LoraConfig(
  task_type=TaskType.SEQ_CLS,
  r=8,               # rank — главный гиперпараметр
  lora_alpha=32,     # scaling = alpha / r
  lora_dropout=0.1,
  target_modules=["q_lin", "v_lin"],  # к каким слоям применяем
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# trainable: 592,898 / 66,955,012 = 0.89% — обучаем <1%!

In [ ]:
# --- Trainer — готовый train loop ---
training_args = TrainingArguments(
  output_dir="./results",
  num_train_epochs=3,
  per_device_train_batch_size=16,
  learning_rate=2e-5,
  weight_decay=0.01,
  evaluation_strategy="epoch",
  logging_steps=10,
  # fp16=True,  # если GPU — снижает память в ~2x
)

In [ ]:
trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=val_dataset,
      tokenizer=tokenizer,
  )

trainer.train()


# Deployment

* vLLM -- для серверов
* llama.cpp -- для локальных устройств
* HiAI, NNAPI -- для мобильных NPU

**Задача на дом (по желанию)**

Дан baseline notebook с обучением модели на датасете MNIST цифры. Цель — максимизировать accuracy при ограничении на размер модели (≤ 1MB state_dict).

Что можно делать:
- Менять архитектуру (меньше слоёв, меньше hidden_dim)
- Квантование (FP16, INT8 через torch.quantization)
- Pruning (torch.nn.utils.prune)
- Knowledge distillation (большая модель учит маленькую)

In [ ]:
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
import numpy as np
import sys

# transform = transforms.Compose([
#                        transforms.ToTensor(),
#                        transforms.Normalize((0.1307,), (0.3081,))
#                     ])

train_dataset = MNIST('.', train=True, download=True)
test_dataset = MNIST('.', train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

X, y = next(iter(train_loader))
X = X.numpy()
y = y.numpy()

plt.figure(figsize=(6, 7))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.imshow(X[i].reshape(28, 28), cmap=plt.cm.Greys_r)
    plt.title(y[i])
    plt.axis('off')